# Изображения: гибридная модель с T1/T2 и параллельным обучением на CPU

Отдельный эксперимент: **MNIST, Fashion MNIST, CIFAR-10**, 8 кубитов,
полносвязные слои до и после квантовой схемы. Исходный ноутбук для табличных данных сохранён.
Выполняйте ячейки по порядку; повторное обучение запускает только ячейка «Обучение».

Бэкенды:

- `torch_density`: специализированная матрица плотности 256×256, автодифференцирование PyTorch;
- `pennylane_mixed`: эталонный `default.mixed`, `ThermalRelaxationError`, `backprop`;
- `pennylane_statevector`: идеальный `default.qubit` без T1/T2, для сравнения с шумовыми моделями.

Шум T1/T2 рассчитывается детерминированно через матрицу плотности. Статистика измерений задаётся отдельно числом shots. Seed обучения отвечает за веса и порядок данных, seed измерений — за выборку результатов.
«Точный» здесь относится к эволюции выбранного канала, а не к гарантированной точности классификации.
`float64` полезен для проверки; `float32` экономит память. Быстродействие измеряется ниже.


In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "qml_hybrid").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src" / "qml_hybrid").is_dir(), "Откройте ноутбук из каталога проекта"
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
import json
from dataclasses import replace
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import ipywidgets as w
from IPython.display import display
import matplotlib.pyplot as plt
from qml_hybrid.model import ModelConfig
from qml_hybrid.noisy import NoiseConfig, NoisyHybridClassifier, circuit_schedule
from qml_hybrid.vision_data import load_vision_data, DATASET_CLASSES
from qml_hybrid.vision_training import (CPUConfig, fit_vision, benchmark_cpu,
                                      report_vision_study, load_noisy_checkpoint)
from qml_hybrid.notebook import LayerEditor, plot_run, parse_edges
from qml_hybrid.seeds import parse_seeds
from qml_hybrid.training import seed_everything
plt.rcParams.update({"font.family": "Times New Roman", "axes.unicode_minus": False})
print("CPU:", os.cpu_count(), "| PyTorch:", torch.__version__)

from qml_hybrid.measurements import measurement_description


## Данные и размер изображений

Официальный тест сохраняется отдельно. Для CIFAR-10 используются официальный train
и официальный test, без валидации. Для MNIST и Fashion MNIST валидацию можно включить:
тогда она выделяется из официального train с сохранением долей классов.
Без валидации обучение идёт заданное число эпох; используются последние веса,
ранняя остановка не применяется. Без val тест оценивается после каждой эпохи:
его потери и точность сохраняются в истории и выводятся для наблюдения за обучением.
Подвыборки также стратифицированы.
Изображения уменьшаются билинейно, каналы CIFAR-10 сохраняются; пиксели переводятся
в диапазон [-1, 1]. Никакие статистики по тесту не подбираются.

По умолчанию включён быстрый пробный режим. Снимите флажок для полного датасета.
Размер 14×14 даёт 196 признаков для серых изображений и 588 для цветных.
Для исходного разрешения укажите 28 (MNIST/Fashion) или 32 (CIFAR).
Первая загрузка сохраняет официальные архивы в `data/vision` и проверяет контрольные суммы.

Поле «Сторона, пкс» задаёт высоту и ширину изображения. 14 означает 14×14: 196 признаков для MNIST/Fashion MNIST, 588 для трёх каналов CIFAR-10.

Флажок «CIFAR-10: оттенки серого» преобразует RGB в один канал до изменения размера. Для 32×32 получится 1024 признака вместо 3072. После переключения заново загрузите данные и создайте модель; Conv2d из примера возьмёт число каналов из metadata.


In [ ]:
dataset = w.Dropdown(options=list(DATASET_CLASSES), value="CIFAR-10", description="Датасет:")
image_size = w.BoundedIntText(value=32, min=2, max=32, description="Сторона, пкс:", style={"description_width": "initial"}, tooltip="Изображение будет уменьшено до указанного числа пикселей по каждой стороне: 14 означает 14×14")
grayscale = w.Checkbox(value=False, description="CIFAR-10: оттенки серого", style={"description_width": "initial"})
small_data = w.Checkbox(value=True, description="Пробная стратифицированная подвыборка")
train_limit = w.BoundedIntText(value=50000, min=10, max=60000, description="Обучение:")
val_limit = w.BoundedIntText(value=250, min=10, max=10000, description="Валидация:")
test_limit = w.BoundedIntText(value=10000, min=10, max=10000, description="Тест:")
split_seed = w.IntText(value=42, description="Seed разбиения:", style={"description_width": "initial"})
val_size = w.BoundedFloatText(value=0.15, min=0.01, max=0.5, description="Доля валидации:", style={"description_width": "initial"})
use_validation = w.Checkbox(value=True, description="Выделять валидацию из train")
split_note = w.HTML()
_validation_preferences = {"MNIST": True, "Fashion MNIST": True}

def _sync_validation_controls(*_):
    is_cifar = dataset.value == "CIFAR-10"
    grayscale.disabled = not is_cifar
    use_validation.disabled = is_cifar
    enabled = use_validation.value and not is_cifar
    val_size.disabled = val_limit.disabled = not enabled
    split_note.value = ("<b>Train + val + test:</b> лучшая эпоха выбирается по val."
                        if enabled else "<b>Train + test:</b> заданное число эпох, последние веса; test после каждой эпохи.")

def _validation_changed(change):
    if dataset.value != "CIFAR-10":
        _validation_preferences[dataset.value] = change["new"]
    _sync_validation_controls()

def _dataset_split_changed(change):
    use_validation.value = _validation_preferences.get(change["new"], False)
    _sync_validation_controls()

use_validation.observe(_validation_changed, names="value")
dataset.observe(_dataset_split_changed, names="value")
_dataset_split_changed({"new": dataset.value})
display(w.VBox([dataset, image_size, grayscale, small_data, w.HBox([train_limit, val_limit, test_limit]),
                split_seed, use_validation, val_size, split_note]))


In [ ]:
import importlib
import qml_hybrid.vision_data as vision_data_module

# Подхватить поддержку val_size=0 и при уже работающем ядре.
importlib.reload(vision_data_module)
load_vision_data = vision_data_module.load_vision_data

data = load_vision_data(dataset.value, ROOT / "data" / "vision", image_size=image_size.value,
                        seed=split_seed.value, grayscale=grayscale.value and dataset.value == "CIFAR-10",
                        val_size=val_size.value if use_validation.value and dataset.value != "CIFAR-10" else 0,
                        train_limit=train_limit.value if small_data.value else 0,
                        val_limit=val_limit.value if small_data.value else 0,
                        test_limit=test_limit.value if small_data.value else 0)
print("Признаков:", data.train_x.shape[1], "| Обучение / валидация / тест:",
      len(data.train_y), len(data.val_y), len(data.test_y))
display(data.distribution())


## Архитектура и физика шума

Для каждого кубита можно задать одно общее время или восемь значений через запятую.
Все времена **в микросекундах**, требуется $0<T_2\leq 2T_1$.
При нулевой температуре $\rho_{11}(t)=\rho_{11}(0)e^{-t/T_1}$,
$\rho_{01}(t)=\rho_{01}(0)e^{-t/T_2}$.
Это учитывает вклад релаксации в T2: дополнительно применять независимое затухание
с тем же T2 нельзя. Для чистой дефазировки $1/T_\varphi=1/T_2-1/(2T_1)$.

Один слой вращений одного типа на восьми кубитах выполняется параллельно.
Двухкубитные операции выполняются последовательно. После каждого такого временного
интервала шум применяется **ко всем кубитам**, включая простаивающие.
Учитываются кодирование, H, повторная загрузка признаков и задержка перед измерением.
Все RX/RY/RZ используют заданную длительность однокубитной операции.
Это дискретная модель «идеальная операция, затем шум», нулевая температура,
независимые марковские каналы; ошибки считывания, перекрёстные помехи и шум импульса
в эту версию не включены. Параметры по умолчанию — пример, замените их калибровкой устройства.

[Канал PennyLane](https://docs.pennylane.ai/en/stable/code/api/pennylane.ThermalRelaxationError.html).

Для идеального вектора состояния T1/T2 отключаются: релаксация и дефазировка в общем случае создают смешанное состояние, которому нужна матрица плотности. При возврате к шумовому бэкенду прежние T1/T2 и выбор включения шума восстанавливаются.

Редактор задаёт только скрытые слои. Для одного скрытого слоя `(32,)` используется одна активация: первый элемент `("relu", "tanh")`, то есть `relu`; второй элемент этого списка не используется. Активация выхода энкодера `tanh` задаётся отдельно полем `latent_activation` конфигурации модели.


### Число измерений (shots)

Поле **«Измерений (shots)»**: `0` — аналитическое среднее (`shots=None`);
положительное число, например `1024`, — столько повторений измерения на каждый объект.
Оценка Pauli Z равна $(N_0-N_1)/N$; для X/Y измерения выполняются в соответствующем базисе.
Все 8 кубитов измеряются совместно, один набор shots даёт 8 средних и сохраняет корреляции.
На векторе состояния shots моделируют статистику измерений идеальной схемы;
на матрице плотности — статистику измерений состояния с T1/T2.

Способ расчёта градиента выбирается в поле **«Режим измерений»**. Parameter-shift использует дополнительные сдвинутые схемы; два новых режима используют backprop через аналитическую симуляцию.
Параметры сохраняются в checkpoint, конфигурациях серий и метриках.
Повторная оценка может отличаться из-за статистического шума. «Seed измерений» фиксирует
поток случайных измерений; CPU-процессы получают разные подпотоки. Для воспроизводимости
полного запуска сохраняйте также число процессов, микробатч и порядок вычислений.


### Режим измерений и градиента

- **Выборочный forward + аналитический Jacobian** (`sampled_backprop`): на вход
  классической головы поступают настоящие совместные измерения восьми кубитов.
  Обратный проход использует Jacobian точного среднего, без parameter-shift.
  Это быстрый суррогатный градиент; он в общем случае смещён относительно градиента
  математического ожидания нелинейной конечной-shot функции потерь.
- **Аналитическое обучение, shots при оценке** (`analytic_train`): `model.train()`
  использует точные средние и backprop; `model.eval()` использует заданные shots.
  **Валидация и тест оба выборочные**, поэтому early stopping видит шумную оценку.
  `torch.no_grad()` сам по себе не переключает режим измерений.
- **Выборочные измерения + parameter-shift** (`parameter_shift`): прежний режим,
  выборочные средние и Jacobian через сдвинутые схемы. Оценка квантового Jacobian
  не гарантирует несмещённости градиента полной нелинейной конечной-shot функции потерь.

Все режимы поддерживают state vector и оба бэкенда матрицы плотности с T1/T2.
При 0 shots все режимы аналитические. Выбор режима, shots и seed входят в checkpoint
и протокол серии; разные режимы не объединяются в одну статистику.


In [ ]:
pre_editor = LayerEditor("Слои до квантовой части", (256, 128, 32), ("relu",  "relu", "tanh"))
post_editor = LayerEditor("Слои после квантовой части", (16,), ("relu",))
depth = w.BoundedIntText(value=4, min=1, max=20, description="Квант. слоёв:")
embedding = w.Dropdown(options=["RY", "RX", "RZ"], value="RX", description="Кодирование:")
rotations = w.Text(value="RY,RX", description="Вращения:")
topology = w.Dropdown(options=[ ("Линия", "linear"), ("Два ряда", "brickwork"), ("Три ряда: чётные–нечётные–чётные", "brickwork3"),("Кольцо", "ring"), ("Полная", "full"), ("Без связей", "none"), ("Свои связи", "custom")], description="Связи:")
custom_edges = w.Text(value="[[0,1],[1,2],[2,7]]", description="Свои связи:", layout=w.Layout(width="650px"))
entangler = w.Dropdown(options=["CZ","CNOT",  "iSWAP"], description="Вентиль:")
measurement = w.Dropdown(options=["Z", "X", "Y"], description="Измерение:")
hadamard = w.Checkbox(value=False, description="Начальные H")
reupload = w.Checkbox(value=False, description="Повторное кодирование")
backend = w.Dropdown(options=[("PyTorch: матрица плотности", "torch_density"), ("PennyLane: матрица плотности", "pennylane_mixed"), ("PennyLane: вектор состояния", "pennylane_statevector"), ("CustomDevice C++: матрица плотности", "custom_cpp_dm"), ("CustomDevice C++: вектор состояния", "custom_cpp_sv")], layout=w.Layout(width="500px"), description="Бэкенд:")
custom_device_path = w.Text(value=str(ROOT / "native" / "custom_device"),
    description="CustomDevice:", style={"description_width": "initial"}, layout=w.Layout(width="850px"))
precision = w.Dropdown(options=["float32", "float64"], description="Точность:")
noise_on = w.Checkbox(value=True, description="Шум T1/T2 включён")
t1 = w.Text(value="14", description="T1, мкс:", layout=w.Layout(width="650px"))
t2 = w.Text(value="4.33", description="T2, мкс:", layout=w.Layout(width="650px"))
single_time = w.BoundedFloatText(value=0.05, min=0, max=1e6, description="1-куб., мкс:")
two_time = w.BoundedFloatText(value=0.05, min=0, max=1e6, description="2-куб., мкс:")
idle_time = w.BoundedFloatText(value=0.0, min=0, max=1e6, description="Ожидание, мкс:")
display(pre_editor.view, post_editor.view)
display(w.VBox([depth, embedding, rotations, topology, custom_edges, entangler, measurement, hadamard, reupload,
                backend, custom_device_path, precision, noise_on, t1, t2, w.HBox([single_time, two_time, idle_time])]))

# При выборе идеального бэкенда значения T1/T2 сохраняются; шум отключается явно.
_previous_noise = {"enabled": noise_on.value}
def sync_backend_controls(change=None):
    ideal = backend.value in ("pennylane_statevector", "custom_cpp_sv")
    custom_device_path.disabled = not backend.value.startswith("custom_cpp_")
    if ideal and not noise_on.disabled:
        _previous_noise["enabled"] = noise_on.value
        noise_on.value = False
        noise_on.disabled = True
    elif not ideal and noise_on.disabled:
        noise_on.disabled = False
        noise_on.value = _previous_noise["enabled"]
    for control in (t1, t2, single_time, two_time, idle_time):
        control.disabled = ideal

backend.observe(sync_backend_controls, names="value")
sync_backend_controls()
display(w.HTML("<b>Редакторы задают скрытые слои.</b> Число входных признаков определяется изображением; "
               "выход энкодера всегда содержит 8 параметров углового кодирования. "
               "После измерений получаются 8 признаков; выход классификатора — 10 классов."))

measurement_shots = w.BoundedIntText(value=1000, min=0, max=10000000, description="Измерений (shots):",
    style={"description_width": "initial"}, tooltip="0 — аналитическое среднее; например, 1024 — среднее по 1024 измерениям на объект")
measurement_seed = w.BoundedIntText(value=42, min=0, max=2**32-1, description="Seed измерений:",
    style={"description_width": "initial"})
display(w.HBox([measurement_shots, measurement_seed]))
measurement_mode = w.Dropdown(options=[
    ("Выборочные измерения + parameter-shift", "parameter_shift"),
    ("Выборочный forward + аналитический Jacobian", "sampled_backprop"),
    ("Аналитическое обучение, shots при оценке", "analytic_train"),
    ("REINFORCE: loss конечных shots", "reinforce"),
], value="parameter_shift", description="Режим измерений:",
   style={"description_width": "initial"}, layout=w.Layout(width="850px"))
display(measurement_mode)
display(w.HTML("<b>Выборочный forward + аналитический Jacobian:</b> реальные конечные измерения на входе "
               "головы, быстрый суррогатный backprop. <br><b>Аналитическое обучение:</b> точные средние "
               "при обучении; заданные shots при валидации и тестировании. <br>"
               "0 shots сохраняет аналитический режим во всех случаях. Настройки применяются при создании модели."))


In [ ]:
gate_errors_on = w.Checkbox(value=False, description="Фиксированные ошибки вентилей")
single_fidelity = w.BoundedFloatText(value=99.0, min=100/3, max=100,
    description="Fidelity 1-куб., %:", style={"description_width": "initial"})
two_fidelity = w.BoundedFloatText(value=90.0, min=20, max=100,
    description="Fidelity 2-куб., %:", style={"description_width": "initial"})
gate_error_seed = w.BoundedIntText(value=42, min=0, max=2**32-1,
    description="Seed ошибок:", style={"description_width": "initial"})
display(w.VBox([gate_errors_on, single_fidelity, two_fidelity, gate_error_seed]))

trainable_embedding = w.Checkbox(value=True, disabled=True, description="Кодирование: отдельный параметр каждого поворота",
    style={"description_width": "initial"}, layout=w.Layout(width="500px"))
display(trainable_embedding)



### REINFORCE для конечного-shot loss

Режим «REINFORCE: loss конечных shots» оценивает градиент ожидаемой кросс-энтропии
после головы при конечных измерениях. Используются log-вероятности всех выбранных
совместных битовых строк и baseline — loss головы на аналитическом среднем,
не зависящий от текущих измерений. Градиент log-вероятностей вычисляется backprop
аналитического симулятора. Baseline не требует второй квантовой симуляции.
Дисперсия оценки может быть большой; несмещённость не гарантирует лучшей сходимости.
При 0 shots сохраняется аналитическое обучение. Встроенный тренер подключает loss
автоматически; в собственном цикле используйте `classification_loss` из
`qml_hybrid.measurements` вместо обычного `cross_entropy`.


## Классическая архитектура кодом: полносвязные и свёрточные слои

В ячейке ниже выберите `classical_source = "code"`, чтобы использовать обычный
`torch.nn.Sequential`. Режим `"widgets"` сохраняет прежние полносвязные редакторы.
В кодовом режиме энкодер должен явно вернуть **8 чисел**, а голова — логиты классов.
Активация `Tanh` перед квантовой частью тоже указывается явно.

Пример содержит Conv2d, активацию, pooling и полносвязные слои. `image_shape`
автоматически восстанавливает `(каналы, высота, ширина)` из плоских входов.
Задавайте kernel_size, stride, padding, число каналов и нейронов прямо в коде.
Голова может быть другой `Sequential`; для неё вход имеет размер 8.

Архитектура сериализуется, чтобы те же слои создавались в CPU-процессах,
при каждом seed и при загрузке checkpoint. Модули в примере задают структуру;
их случайные веса не копируются в модель. Поддерживаются Linear, Conv2d,
MaxPool2d, AvgPool2d, AdaptiveAvgPool2d, Flatten, Unflatten, стандартные активации
и вложенный Sequential. Пользовательские классы, BatchNorm и Dropout пока не
поддерживаются этим переносимым форматом. Перед запуском проверяются размеры.


In [ ]:
classical_source = "widgets"  # "code" — архитектура из функции ниже

In [ ]:
# Обновить классы после изменений кода без перезапуска ядра Jupyter.
import importlib
import qml_hybrid.measurements as measurement_module
import qml_hybrid.model as model_module
import qml_hybrid.noisy as noisy_module
import qml_hybrid.vision_training as vision_module
import qml_hybrid.seeds as seed_module
import qml_hybrid.notebook as notebook_module
import qml_hybrid.classical as classical_module
import qml_hybrid.gate_errors as gate_error_module

importlib.reload(measurement_module)
importlib.reload(classical_module)
importlib.reload(gate_error_module)
import qml_hybrid.custom_cpp as custom_cpp_module
importlib.reload(custom_cpp_module)
importlib.reload(model_module)
importlib.reload(noisy_module)
importlib.reload(seed_module)
importlib.reload(vision_module)
importlib.reload(notebook_module)
plot_run = notebook_module.plot_run
ModelConfig = model_module.ModelConfig
NoiseConfig = noisy_module.NoiseConfig
NoisyHybridClassifier = noisy_module.NoisyHybridClassifier
circuit_schedule = noisy_module.circuit_schedule
CPUConfig = vision_module.CPUConfig
fit_vision = vision_module.fit_vision
benchmark_cpu = vision_module.benchmark_cpu
report_vision_study = vision_module.report_vision_study
load_noisy_checkpoint = vision_module.load_noisy_checkpoint

def times(text):
    values = tuple(float(value.strip()) for value in text.split(","))
    return values * 8 if len(values) == 1 else values

pre_widths, pre_activations = pre_editor.values()
post_widths, post_activations = post_editor.values()
if classical_source == "widgets":
    classical_settings = dict(pre_layers=pre_widths, pre_activations=pre_activations,
                              post_layers=post_widths, post_activations=post_activations)
elif classical_source == "code":
    classical_settings = classical_from_code()
elif classical_source == "saved":
    classical_settings = classical_saved_config
else:
    raise ValueError("classical_source: widgets, code или saved")
trainable_embedding.value = True
model_config = ModelConfig(n_features=data.train_x.shape[1], n_classes=10,
    **classical_settings,
    quantum_depth=depth.value, embedding=embedding.value,
    trainable_embedding=True, embedding_parameterization="per_rotation",
    rotation_gates=tuple(g.strip().upper() for g in rotations.value.split(",") if g.strip()),
    entanglement=topology.value, custom_edges=parse_edges(custom_edges.value),
    entangler=entangler.value, measurement=measurement.value,
    initial_hadamard=hadamard.value, reupload=reupload.value,
    shots=measurement_shots.value or None, measurement_seed=measurement_seed.value,
    measurement_mode=measurement_mode.value)
noise_config = NoiseConfig(t1_us=times(t1.value), t2_us=times(t2.value),
    single_gate_us=single_time.value, two_gate_us=two_time.value, readout_idle_us=idle_time.value,
    enabled=noise_on.value, backend=backend.value, precision=precision.value,
    gate_errors=gate_errors_on.value, single_gate_fidelity=single_fidelity.value/100,
    two_gate_fidelity=two_fidelity.value/100, gate_error_seed=gate_error_seed.value,
    custom_device_path=custom_device_path.value if backend.value.startswith("custom_cpp_") else "")
seed_everything(11)
model = NoisyHybridClassifier(model_config, noise_config)
print("Классическая архитектура:", classical_source)
print("Входных признаков:", model_config.n_features, "| Выход энкодера: 8 | Классов:", model_config.n_classes)
print(model.encoder)
print(model.head)
print("Длительность схемы, мкс:", sum(step[2] for step in circuit_schedule(model_config, noise_config)))
if noise_config.backend in ("pennylane_statevector", "custom_cpp_sv"):
    print("Вектор состояния: 256 комплексных амплитуд на объект; T1/T2 отключены.")
else:
    print("Элементов матрицы плотности на объект:", 256 * 256)

print("Измерение:", measurement_description(model_config))
if noise_config.backend.startswith("custom_cpp_"):
    print("CustomDevice: complex128 внутри C++; аналитический обратный проход C++/NumPy.")


In [ ]:
# Подсчёт обучаемых параметров по частям текущей модели
parts = {
    "encoder": model.encoder,
    "quantum": model.quantum,
    "head": model.head,
}

for name, module in parts.items():
    n = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: {n}")

classical_total = (
    sum(p.numel() for p in model.encoder.parameters() if p.requires_grad) +
    sum(p.numel() for p in model.head.parameters() if p.requires_grad)
)
quantum_total = sum(p.numel() for p in model.quantum.parameters() if p.requires_grad)

print(f"classical_total = {classical_total}")
print(f"quantum_total = {quantum_total}")
print(f"total_trainable = {classical_total + quantum_total}")

In [ ]:
28*28*256

## Квантовая цепочка

Схема созданной выше модели: 8 кубитов в состоянии $|0\rangle$, без численных углов,
с увеличенными подписями и шрифтом Times New Roman. Обучение не запускается.
По умолчанию показаны вентили; `show_noise=True` также отображает каналы T1/T2,
если шум включён в модели. После изменения архитектуры сначала создайте модель заново.


In [ ]:
import importlib
import qml_hybrid.notebook as notebook_helpers

importlib.reload(notebook_helpers)
draw_vision_circuit = notebook_helpers.draw_vision_circuit

output_path = ROOT / "runs" / "quantum_circuit_vision.png"
figure, _ = draw_vision_circuit(model, output_path, show_noise=False)
display(figure)
print(f"Сохранено: {output_path}")
plt.close(figure)


## CPU: процессы, микробатчи и общий шаг оптимизатора

`Размер батча` — **общее** число объектов на один шаг Adam независимо от числа процессов.
В режиме «Части батча» общий батч делится между процессами. В режиме «Очередь микробатчей»
процессы получают несколько мелких задач из того же общего батча.
Во всех случаях градиенты суммируются по объектам и делятся на общий размер батча.
Один шаг Adam выполняется после завершения всех задач. При неравных частях учитывается
число объектов, а не число процессов. Это сохраняет математический смысл обычного обучения.

Каждый процесс обрабатывает данные микробатчами и сразу освобождает граф после backward.
Веса и данные передаются через общую память CPU, работники читают веса, основной процесс
изменяет их только после барьера. Функции процессов вынесены в импортируемый модуль,
что позволяет запуск из Jupyter под Windows.

Начните с 1–2 процессов, 1 потока и микробатча 1–2. Число процессов × потоки не должно
превышать доступные CPU. Память под графы растёт с числом процессов и размером микробатча;
матрица 256×256 — лишь состояние, backward требует гораздо больше памяти.
Ускорение не гарантируется: на короткой схеме затраты IPC могут перевесить выигрыш.
Сначала измерьте время на своём CPU. Локальные процессы используют одно физическое устройство;
это не распределение по нескольким компьютерам.


In [ ]:
workers = w.BoundedIntText(value=20, min=1, max=os.cpu_count() or 1, description="Процессов:")
threads = w.BoundedIntText(value=1, min=1, max=os.cpu_count() or 1, description="Потоков/проц.:")
microbatch = w.BoundedIntText(value=2, min=1, max=256, description="Микробатч:")
parallel_mode = w.Dropdown(options=[("Части батча", "split_batch"), ("Очередь микробатчей", "batch_queue")], description="Режим:")
batch_size = w.BoundedIntText(value=5000, min=1, max=60000, description="Размер батча:")
epochs = w.BoundedIntText(value=150, min=1, max=10000, description="Эпох:")
patience = w.BoundedIntText(value=100, min=0, max=10000, description="Терпение:")
min_delta = w.BoundedFloatText(value=0.0, min=0.0, max=100, description="Мин. улучшение:")
learning_rate = w.FloatText(value=6e-3, description="Шаг классич.:")
quantum_rate = w.FloatText(value=6e-3, description="Шаг квант.:")
regularization_classical = w.BoundedFloatText(value=0.0, min=0, max=1e6,
    description="L2 классики:", style={"description_width": "initial"})
regularization_quantum = w.BoundedFloatText(value=0.0, min=0, max=1e6,
    description="Штраф углов:", style={"description_width": "initial"})
regularization_embedding = w.BoundedFloatText(value=0.0, min=0, max=1e6,
    description="L2 масштабов:", style={"description_width": "initial"})
use_quantum = w.Checkbox(value=True, description="Квантовая часть включена")
train_classical = w.Checkbox(value=True, description="Обучать классическую часть")
train_quantum = w.Checkbox(value=True, description="Обучать квантовую часть")
seeds = w.Text(value="11", description="Seed обучения:", style={"description_width": "initial"}, tooltip="Числа для инициализации весов и перемешивания данных. Например: 11,22,33. Разбиение задаётся отдельно")
output_name = w.Text(value="noisy_vision", description="Имя серии:")
display(w.VBox([w.HBox([workers, threads, microbatch]), parallel_mode, batch_size, epochs,
                patience, min_delta, learning_rate, quantum_rate,
                regularization_classical, regularization_quantum, regularization_embedding,
                w.HTML("0 отключает штраф. L2 классики — веса Linear/Conv2d без смещений; "
                       "углы — периодический штраф; масштабы кодирования стягиваются к 1."),
                use_quantum, train_classical,
                train_quantum, seeds, output_name]))
def current_cpu():
    return CPUConfig(workers=workers.value, threads_per_worker=threads.value,
                     microbatch_size=microbatch.value, parallel_mode=parallel_mode.value)

def current_regularization():
    return dict(classical_l2=regularization_classical.value,
                quantum_periodic=regularization_quantum.value,
                embedding_l2=regularization_embedding.value)


## Обучение

In [ ]:
continue_current_model = False
selected_seeds = parse_seeds(seeds.value)
if continue_current_model and len(selected_seeds) != 1:
    raise ValueError("Продолжение текущей модели допускает одно начальное значение")
if Path(output_name.value).name != output_name.value or output_name.value in ("", ".", ".."):
    raise ValueError("Имя серии должно быть простым именем каталога")
study_folder = ROOT / "runs" / output_name.value / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
results = []
for seed in selected_seeds:
    if not continue_current_model:
        seed_everything(seed)
        model = NoisyHybridClassifier(model_config, noise_config)
    model.set_mode(quantum=use_quantum.value, train_classical=train_classical.value, train_quantum=train_quantum.value)
    result = fit_vision(model, data, study_folder / f"seed_{seed}", cpu=current_cpu(),
        epochs=epochs.value, batch_size=batch_size.value, patience=patience.value, min_delta=min_delta.value,
        lr=learning_rate.value, quantum_lr=quantum_rate.value, seed=seed,
        regularization=current_regularization(),
        initialization="continued" if continue_current_model else "independent")
    results.append(result)
    plot_run(result)
print("Результаты:", study_folder)


## Сбор начальных значений и графики разброса


In [ ]:
import importlib
import qml_hybrid.plot_style as plot_style_module
importlib.reload(plot_style_module)
import qml_hybrid.seeds as seed_reports
import qml_hybrid.vision_training as vision_training

importlib.reload(seed_reports)
importlib.reload(vision_training)

study_folder = Path(
    ROOT / "runs/noisy_vision/20261021_233017_092277_sv_50k_10k_10k_copy"
    # ROOT / "runs/noisy_vision/20261022_142358_183767"
    # ROOT / "runs/noisy_vision/20261017_123726_664069_fmnist_sv"
)

collect_paths = sorted(
    p.parent for p in study_folder.glob("seed_*/metrics.json")
)

band = "sd"  # стандартное отклонение между seed
# band = "ci95"  # альтернативно: 95%-й доверительный интервал среднего

print("Найдено запусков:", len(collect_paths))
assert collect_paths, "Папки seed_* с metrics.json не найдены"

legend_loc = "lower right"  # upper right / upper left / lower right / lower left / best; None — скрыть
legend_on = "right"  # left — слева; right — справа; both — оба; none — скрыть
legend_ncol = 1  # число колонок легенды

panel_positions = ((-0.12, 1.04), (-0.12, 1.04))  # (x, y) для (а) и (б); None — скрыть

summary = vision_training.report_vision_study(
    collect_paths,
    study_folder / "report_selected",
    band=band,
    seeds=[11,22, 44,55, 66],  # укажите нужные seed
    legend_loc=legend_loc, legend_on=legend_on,
    legend_ncol=legend_ncol,
    allow_mixed_protocols=True,
    panel_positions=panel_positions,
)
display(summary)

In [ ]:
# Графики из сохранённых результатов. Ячейки обучения выполнять не нужно.
from pathlib import Path
import importlib
import qml_hybrid.plot_style as plot_style_module
importlib.reload(plot_style_module)
import qml_hybrid.vision_training as vision_training

# Однократное обновление сборщика в уже работающем ядре.
importlib.reload(vision_training)
CPUConfig = vision_training.CPUConfig
fit_vision = vision_training.fit_vision
benchmark_cpu = vision_training.benchmark_cpu
report_vision_study = vision_training.report_vision_study
load_noisy_checkpoint = vision_training.load_noisy_checkpoint

import qml_hybrid.vision_plots as vision_plots
importlib.reload(vision_plots)
plot_root = Path.cwd().resolve()
if not (plot_root / "src" / "qml_hybrid").is_dir():
    plot_root = plot_root.parent

# Ключ — подпись на графике. Значение — папка серии или список папок seed_*.
# Для сравнения классики и гибрида укажите соответствующие папки и подписи.
comparison_studies = {
    # "Fashion MNIST, T1/T2": plot_root / "runs/noisy_vision/20261017_150125_863721_fmnist_noise_23-50",
    "Fashion MNIST, SV": plot_root / "runs/noisy_vision/20261017_123726_664069_fmnist_sv",
}
comparison_seeds = None     # все seed; либо [11, 22, 33, 44]
comparison_band = "sd"     # "sd" или "ci95"
comparison_split = "auto"  # val при наличии, иначе test; можно явно "val" или "test"
comparison_epochs = None

# comparison_xlim = None   # диапазон оси эпох: например, (0, 40) или (10, 100)
comparison_xlim = None

comparison_ylim_loss = None
comparison_ylim_accuracy = None

# Ключ из comparison_studies ИЛИ отдельная папка серии / список папок seed_*.
# Например: plot_root / "runs/noisy_vision/имя_измерения". None — убрать ориентир.
comparison_test_reference = None
# comparison_test_reference = plot_root / "runs/noisy_vision/20261021_233017_092277_sv_50k_10k_10k_fine"
comparison_test_epoch = 100  # точная эпоха ориентира; None — последняя доступная внутри xlim
comparison_reference_seeds = None  # все seed или [11, 22, 33]
comparison_reference_label = "Тестовая выборка, предел (50.000 изображений)"

comparison_legend_loc = "lower left" 
comparison_legend_on = "left"
comparison_legend_ncol = 1 

comparison_allow_mixed_protocols = False

comparison_panel_positions = None #((-0.12, 1.04), (-0.12, 1.04))  # (x, y) для (а) и (б); None — скрыть

comparison = vision_plots.plot_vision_comparison(
    comparison_studies,
    plot_root / "runs/noisy_vision/plots_comparison_fmnist_noise_sv",
    seeds=comparison_seeds, band=comparison_band,
    eval_split=comparison_split, max_epochs=comparison_epochs, xlim=comparison_xlim,
    ylim_loss=comparison_ylim_loss, ylim_accuracy=comparison_ylim_accuracy,
    test_reference=comparison_test_reference, test_epoch=comparison_test_epoch,
    reference_seeds=comparison_reference_seeds, reference_label=comparison_reference_label,
    legend_loc=comparison_legend_loc, legend_on=comparison_legend_on,
    legend_ncol=comparison_legend_ncol,
    allow_mixed_protocols=comparison_allow_mixed_protocols,
    panel_positions=comparison_panel_positions,
)
display(comparison["curves"])
print("Графики PNG/SVG и данные CSV:", comparison["output"])


In [ ]:
# Выполните после ячейки с графиками. Обучение повторять не требуется.
importlib.reload(vision_plots)
best_metrics = vision_plots.comparison_best_metrics(comparison)
display(best_metrics.style.format({
    "Величина разброса (±)": "{:.4f}",
    "Значение": "{:.4f}", "Нижняя граница": "{:.4f}", "Верхняя граница": "{:.4f}",
}, na_rep="—"))
best_metrics.to_csv(Path(comparison["output"]) / "best_metrics.csv", index=False, encoding="utf-8-sig")


In [ ]:
checkpoint_path = ""
if checkpoint_path:
    model, checkpoint = load_noisy_checkpoint(checkpoint_path)
    model_config, noise_config = model.config, model.noise
    backend.value = noise_config.backend
    custom_device_path.value = noise_config.custom_device_path or custom_device_path.value
    precision.value = noise_config.precision
    noise_on.value = noise_config.enabled
    entangler.value = model_config.entangler
    if model_config.encoder_spec is not None or model_config.head_spec is not None:
        classical_source = "saved"
        classical_saved_config = dict(encoder_spec=model_config.encoder_spec,
                                      head_spec=model_config.head_spec, image_shape=model_config.image_shape)
    gate_errors_on.value = noise_config.gate_errors
    single_fidelity.value = 100 * noise_config.single_gate_fidelity
    two_fidelity.value = 100 * noise_config.two_gate_fidelity
    gate_error_seed.value = noise_config.gate_error_seed
    measurement_mode.value = model_config.measurement_mode
    measurement_shots.value = model_config.shots or 0
    measurement_seed.value = model_config.measurement_seed
    trainable_embedding.value = model_config.trainable_embedding
    restored_regularization = checkpoint.get("settings", {}).get("regularization", {})
    regularization_classical.value = restored_regularization.get("classical_l2", 0.0)
    regularization_quantum.value = restored_regularization.get("quantum_periodic", 0.0)
    regularization_embedding.value = restored_regularization.get("embedding_l2", 0.0)
    use_quantum.value = model.use_quantum
    train_classical.value = model.train_classical
    train_quantum.value = model.train_quantum
    print("Восстановлена эпоха:", checkpoint["epoch"])


## QPU time est

In [ ]:
n_shots = 1000

t_PQC = 1200
t_readout = 1000
t_equilibrium = 5000

t_waveform = 10.0 * 1e6

n_1q = 96

t_available = 100 # hours
n_epochs = 40 # проход по всем данным

train_size = 50000


In [ ]:
t_1elem = (n_shots * (t_PQC + t_readout + t_equilibrium) + t_waveform) * (2*n_1q + 1)
print(f'1 element time: {t_1elem * 1e-9:.2f} s')
print(f'train time for epoch: {t_1elem * 1e-9 * train_size / 3600 :.2f} h \n')

print(f'Available time: {t_available:.2f} h')
print(f'In available time, you can train for {t_available / (t_1elem * 1e-9 * train_size / 3600):.2f} epochs. \n')

print(f'elements to match {n_epochs} epochs in available time: {t_available / (n_epochs * (t_1elem * 1e-9 / 3600)):.2f} elements')


